<a href="https://colab.research.google.com/github/ajsarsva/video-captioning-thesis/blob/main/notebooks/day7_8_video_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Master Cell

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

if os.path.exists('/content/video-captioning-thesis'):
    %cd /content/video-captioning-thesis
    !git pull origin main
else:
    !git clone https://github.com/ajsarsva/video-captioning-thesis.git
    %cd /content/video-captioning-thesis

import sys
sys.path.append('/content/video-captioning-thesis/src')

print("✅ Ready!")

Mounted at /content/drive
Cloning into 'video-captioning-thesis'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 37 (delta 12), reused 15 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 637.82 KiB | 23.62 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/video-captioning-thesis
✅ Ready!


In [2]:
!pip install transformers accelerate -q
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


Load BLIP model

In [3]:
from blip_captioner import load_blip_model
processor, model = load_blip_model()

Loading BLIP model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

BLIP model loaded on cuda!


Testing on a Single Video First

In [4]:
from frame_extractor import extract_frames
from strategy_a_uniform import uniform_sampling
from blip_captioner import frames_to_caption

video_path = '/content/drive/MyDrive/thesis-data/videos/video0.mp4'

# Extract frames
frames, fps, total = extract_frames(video_path)
print(f"Total frames: {len(frames)}")

# Apply uniform sampling
keyframes, indices = uniform_sampling(frames, K=8)
print(f"Selected frame indices: {indices}")

# Generate caption
caption = frames_to_caption(keyframes)
print(f"Generated caption: {caption}")

Total frames: 300
Selected frame indices: [0, 42, 85, 128, 170, 213, 256, 299]
Generated caption: a road with a sign on it


Run for All 50 Videos and Saving Results

In [5]:
import json, os, time

video_dir = '/content/drive/MyDrive/thesis-data/videos/'
results = {}
errors = []

start_time = time.time()

for i in range(50):
    video_name = f'video{i}'
    video_path = os.path.join(video_dir, f'{video_name}.mp4')

    try:
        frames, fps, total = extract_frames(video_path)
        keyframes, indices = uniform_sampling(frames, K=8)
        caption = frames_to_caption(keyframes)

        results[video_name] = {
            'caption': caption,
            'keyframe_indices': indices,
            'total_frames': len(frames)
        }

        print(f"video{i}: {caption}")

    except Exception as e:
        print(f"video{i}: ERROR — {e} ❌")
        errors.append(video_name)

elapsed = time.time() - start_time
print(f"\nDone! {len(results)}/50 videos captioned")
print(f"Time taken: {elapsed:.1f} seconds")
print(f"Avg per video: {elapsed/50:.1f} seconds")
print(f"Errors: {errors}")

video0: a road with a sign on it
video1: a person is cooking in a pot
video2: a man holding a sign
video3: a purple and white background with a white stripe
video4: a woman standing in a kitchen with a dog
video5: a baby in a swing with a cat
video6: a cat is standing in the grass near a tree
video7: two cats and a dog eating food out of a bowl
video8: a man sitting on a chair
video9: a man in a suit and tie sitting on a tv screen
video10: a man holding two dogs in his arms
video11: a man with long hair
video12: a man in a suit standing in an office
video13: a man in a top hat and topcoat is standing in front of a giant stone
video14: a man in a black jacket is pointing at a woman
video15: a black and white photo frame with a black background
video16: a computer screen showing a video of a man on a computer
video17: a video of a building being used as a hospital
video18: a man in a suit and tie standing in front of a rack of clothes
video19: a television screen showing a video of a tab

In [6]:
import json

# Load reference captions
with open('/content/drive/MyDrive/thesis-data/MSR_VTT.json', 'r') as f:
    data = json.load(f)

# Get all reference captions for video0
ref_captions = [ann['caption'] for ann in data['annotations']
                if ann['image_id'] == 'video0']

print("=" * 50)
print("GENERATED CAPTION:")
print(caption)  # this is from Step 6
print()
print("REFERENCE CAPTIONS (ground truth):")
for i, ref in enumerate(ref_captions):
    print(f"{i+1}. {ref}")
print("=" * 50)

GENERATED CAPTION:
a soccer player is kicking the ball

REFERENCE CAPTIONS (ground truth):
1. a car is shown
2. a group is dancing
3. a man drives a vehicle through the countryside
4. a man drives down the road in an audi
5. a man driving a car
6. a man is driving a car
7. a man is driving down a road
8. a man is driving in a car as part of a commercial
9. a man is driving
10. a man riding the car speedly in a narrow road
11. a man showing the various features of a car
12. a man silently narrates his experience driving an audi
13. a person is driving his car around curves in the road
14. a person telling about a car
15. guy driving a car down the road
16. man talking about a car while driving
17. the man drives the car
18. the man driving the audi as smooth as possible
19. a man is driving
20. guy driving a car down the road


Save Results to Drive

In [8]:
# Save captions
with open('/content/drive/MyDrive/thesis-data/captions/strategy_A.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Strategy A captions saved to Drive!")

# Preview a few results
print("\nSample captions:")
for vid in ['video0', 'video1', 'video2']:
    print(f"{vid}: {results[vid]['caption']}")

Strategy A captions saved to Drive!

Sample captions:
video0: a road with a sign on it
video1: a person is cooking in a pot
video2: a man holding a sign


Result Comparision

In [9]:
import json

with open('/content/drive/MyDrive/thesis-data/MSR_VTT.json', 'r') as f:
    data = json.load(f)

# Load our generated captions
with open('/content/drive/MyDrive/thesis-data/captions/strategy_A.json', 'r') as f:
    strategy_a = json.load(f)

# Compare first 5 videos
for i in range(5):
    video_id = f'video{i}'

    generated = strategy_a[video_id]['caption']

    refs = [ann['caption'] for ann in data['annotations']
            if ann['image_id'] == video_id]

    print(f"{'='*50}")
    print(f"VIDEO: {video_id}")
    print(f"GENERATED : {generated}")
    print(f"REFERENCE 1: {refs[0]}")
    print(f"REFERENCE 2: {refs[1]}")
    print(f"REFERENCE 3: {refs[2]}")
    print()

VIDEO: video0
GENERATED : a road with a sign on it
REFERENCE 1: a car is shown
REFERENCE 2: a group is dancing
REFERENCE 3: a man drives a vehicle through the countryside

VIDEO: video1
GENERATED : a person is cooking in a pot
REFERENCE 1: in a kitchen a woman adds different ingredients into the pot and stirs it
REFERENCE 2: a woman puts prawns and seasonings into a large pot on a stove
REFERENCE 3: in the kitchen a woman makes a dish by adding ingredients mixing and allowing to boil on flame

VIDEO: video2
GENERATED : a man holding a sign
REFERENCE 1: a guying showing a tool
REFERENCE 2: a man fixes a car
REFERENCE 3: a man holding a combustion leak tester

VIDEO: video3
GENERATED : a purple and white background with a white stripe
REFERENCE 1: a big door is being opened in a video game
REFERENCE 2: a bright light is flashing
REFERENCE 3: a cartoon of a door opening

VIDEO: video4
GENERATED : a woman standing in a kitchen with a dog
REFERENCE 1: a girl wearing a black shirt
REFERENCE 